# Load DIV30 and DIV90 Raw 10x Samples to AnnData

This notebook defines a small in-notebook builder class for raw 10x sample loading. The immediate focus is DIV30, but the same object works for DIV90 or both time points by changing `target_divs`.

The builder reads per-sample 10x matrices from `metadata/div30_div90_sample_id_to_biolabel_map.tsv`, adds sample metadata to `.obs`, makes cell IDs unique by prefixing the 10x barcode with `run_sample_id`, and can return either a standard combined `AnnData` or, if `snapatac2` is available, a backed `AnnDataSet`.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

In [ ]:
@dataclass
class Raw10xAnnDataBuilder:
    """Minimal teaching builder: only methods used in this tutorial."""

    project_root: Path
    target_divs: Sequence[str] = ("DIV30",)
    target_run_sample_ids: Optional[Sequence[str]] = None
    sample_map_name: str = "metadata/div30_div90_sample_id_to_biolabel_map.tsv"
    strict_missing_matrix_dirs: bool = True

    @property
    def sample_map_tsv(self) -> Path:
        """Compute sample map path only (no file read)."""
        return self.project_root / self.sample_map_name

    def sample_table(self) -> pd.DataFrame:
        """Read/filter sample metadata and derive per-sample matrix paths."""
        sample_map = pd.read_csv(self.sample_map_tsv, sep="\t")
        required = {"DIV", "run_sample_id", "biological_label", "per_sample_metrics_csv"}
        missing = required.difference(sample_map.columns)
        if missing:
            raise ValueError(f"Sample map is missing required columns: {sorted(missing)}")

        sample_map = sample_map[sample_map["DIV"].isin(self.target_divs)].copy()

        if self.target_run_sample_ids is not None:
            sample_map = sample_map[sample_map["run_sample_id"].isin(self.target_run_sample_ids)].copy()
            found = set(sample_map["run_sample_id"].astype(str))
            missing_ids = [sid for sid in self.target_run_sample_ids if sid not in found]
            if missing_ids:
                raise ValueError(f"Missing requested run_sample_id values in sample map: {missing_ids}")

        if sample_map.empty:
            raise ValueError("No samples matched target_divs/target_run_sample_ids.")

        sample_map["DIV"] = pd.Categorical(sample_map["DIV"], categories=self.target_divs, ordered=True)
        if self.target_run_sample_ids is not None:
            sample_map["run_sample_id"] = pd.Categorical(
                sample_map["run_sample_id"],
                categories=self.target_run_sample_ids,
                ordered=True,
            )

        sample_map = sample_map.sort_values(["DIV", "run_sample_id"]).reset_index(drop=True)
        sample_map["matrix_dir"] = sample_map["per_sample_metrics_csv"].map(
            lambda p: str(Path(p).parent / "count" / "sample_filtered_feature_bc_matrix")
        )
        return sample_map

    def combined_anndata(self) -> ad.AnnData:
        """Load selected 10x matrices, annotate obs, and concatenate cells."""
        sample_map = self.sample_table()
        component_adatas = []
        missing_dirs = []

        for _, row in sample_map.iterrows():
            matrix_dir = Path(row["matrix_dir"])
            if not matrix_dir.exists():
                missing_dirs.append(str(matrix_dir))
                continue

            one = sc.read_10x_mtx(matrix_dir, var_names="gene_symbols", make_unique=True)
            run_sample_id = str(row["run_sample_id"])

            one.obs_names = [f"{run_sample_id}:{barcode}" for barcode in one.obs_names]
            one.obs["DIV"] = str(row["DIV"])
            one.obs["run_sample_id"] = run_sample_id
            one.obs["biological_label"] = str(row["biological_label"])
            one.obs["matrix_dir"] = str(matrix_dir)
            component_adatas.append(one)

        if missing_dirs:
            message = "Missing per-sample 10x matrix directories:\n" + "\n".join(f" - {p}" for p in missing_dirs)
            if self.strict_missing_matrix_dirs:
                raise FileNotFoundError(message)
            print(message)

        if not component_adatas:
            raise FileNotFoundError("No per-sample 10x matrix directories were found.")

        combined = ad.concat(
            component_adatas,
            axis=0,
            join="outer",
            merge="same",
            fill_value=0,
            index_unique=None,
        )

        if not combined.obs_names.is_unique:
            duplicated = combined.obs_names[combined.obs_names.duplicated()].unique()[:10].tolist()
            raise ValueError(f"Combined AnnData has duplicated obs_names. Examples: {duplicated}")

        combined.obs["DIV"] = pd.Categorical(combined.obs["DIV"], categories=self.target_divs, ordered=True)
        combined.obs["run_sample_id"] = pd.Categorical(
            combined.obs["run_sample_id"],
            categories=sample_map["run_sample_id"].astype(str).tolist(),
            ordered=True,
        )
        combined.obs["biological_label"] = combined.obs["biological_label"].astype("string")
        combined.obs["matrix_dir"] = combined.obs["matrix_dir"].astype("string")

        combined.uns["sample_map_tsv"] = str(self.sample_map_tsv)
        combined.uns["target_divs"] = list(self.target_divs)
        combined.uns["combine_logic"] = (
            "Per-sample 10x matrices were read separately, annotated, barcode-prefixed by "
            "run_sample_id, and concatenated along observations with an outer gene join."
        )
        return combined

## Builder Instantiation Tutorial (Minimal Method Version)

This notebook now uses a simplified class that keeps only the methods needed for your learning path:

1. `sample_map_tsv` (property)
2. `sample_table()`
3. `combined_anndata()`

Methods like `read_one_sample`, `component_anndatas`, `cell_count_tables`, and `anndata_set` were intentionally removed for clarity. Their logic is now either unnecessary for this tutorial or inlined into `combined_anndata()`.

### What happens when you run builder creation

```python
builder = Raw10xAnnDataBuilder(
    project_root=PROJECT_ROOT,
    target_divs=("DIV30",),
    target_run_sample_ids=(
        "9853-MW-1",
        "9853-MW-2",
        "9853-MW-3",
        "9853-MW-4",
        "9853-MW-5",
        "9853-MW-6",
    ),
)
```

Only dataclass initialization happens:

1. A `Raw10xAnnDataBuilder` instance is created.
2. Fields are assigned (`project_root`, `target_divs`, `target_run_sample_ids`, defaults).
3. No file I/O happens yet.
4. No 10x matrix loading happens yet.

### What happens when you run `builder.sample_table()`

1. Reads `metadata/div30_div90_sample_id_to_biolabel_map.tsv`.
2. Validates required columns exist.
3. Filters rows by `target_divs`.
4. Filters rows by `target_run_sample_ids` and errors if requested IDs are missing.
5. Builds `matrix_dir` for each sample.
6. Returns a filtered metadata table.

### What happens when you run `builder.combined_anndata()`

This is the first heavy method. Internally it:

1. Calls `sample_table()` to get selected sample metadata.
2. Loops through each row and checks whether each `matrix_dir` exists.
3. Uses `sc.read_10x_mtx(...)` to load each sample's 10x matrix.
4. Prefixes each barcode with `run_sample_id` so cell IDs are globally unique.
5. Adds per-cell metadata columns in `.obs`:
   - `DIV`
   - `run_sample_id`
   - `biological_label`
   - `matrix_dir`
6. Concatenates per-sample AnnData objects with `ad.concat(..., axis=0, join="outer")`.
7. Validates uniqueness of `obs_names`.
8. Casts `.obs` columns to stable types/categories.
9. Stores provenance in `.uns` (`sample_map_tsv`, `target_divs`, `combine_logic`).
10. Returns one combined `AnnData` object.

### Mental model

- `builder = ...` is configuration.
- `sample_table()` is metadata I/O.
- `combined_anndata()` is matrix I/O + full object construction.

In [ ]:
# Tutorial cell: instantiate Raw10xAnnDataBuilder with explicit, traceable intent.
#

# -----------------------------------------------------------------------------
# 1) Find project root (pure path discovery; no data loading yet)
# -----------------------------------------------------------------------------
# This function walks upward from the current working directory and returns the
# first parent folder containing the sample map TSV used by the builder.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "metadata" / "div30_div90_sample_id_to_biolabel_map.tsv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing metadata/div30_div90_sample_id_to_biolabel_map.tsv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# -----------------------------------------------------------------------------
# 2) Instantiate the builder (configuration only)
# -----------------------------------------------------------------------------
# IMPORTANT: this line does NOT read TSV files and does NOT load 10x matrices.
# It only constructs a Python object that stores configuration for later calls.
builder = Raw10xAnnDataBuilder(
    # Base folder used to resolve metadata/div30_div90_sample_id_to_biolabel_map.tsv
    project_root=PROJECT_ROOT,

    # Restrict downstream sample selection to DIV30 only.
    # A one-item tuple must include a trailing comma in Python: ("DIV30",)
    target_divs=("DIV30",),

    # Restrict to exactly these run_sample_id values, preserving this order.
    # This affects downstream filtering and categorical ordering in obs metadata.
    target_run_sample_ids=(
        "9853-MW-1",
        "9853-MW-2",
        "9853-MW-3",
        "9853-MW-4",
        "9853-MW-5",
        "9853-MW-6",
    ),
    # sample_map_name and strict_missing_matrix_dirs are omitted here, so defaults apply:
    # sample_map_name="metadata/div30_div90_sample_id_to_biolabel_map.tsv"
    # strict_missing_matrix_dirs=True
)

# -----------------------------------------------------------------------------
# 3) Introspect what was created right now
# -----------------------------------------------------------------------------
# `builder` is an INSTANCE of Raw10xAnnDataBuilder.
# The dataclass-generated __init__ has assigned fields and returned.
# No heavy I/O has happened yet.
print("Type(builder):", type(builder))
print("Builder configuration currently stored:")
print("  project_root:", builder.project_root)
print("  target_divs:", builder.target_divs)
print("  target_run_sample_ids:", builder.target_run_sample_ids)
print("  sample_map_name (default):", builder.sample_map_name)
print("  strict_missing_matrix_dirs (default):", builder.strict_missing_matrix_dirs)

# Accessing sample_map_tsv computes/returns a Path from existing fields.
# This still does not read the TSV contents; it only builds the path string/object.
print("  sample_map_tsv path:", builder.sample_map_tsv)

# -----------------------------------------------------------------------------
# 4) First operation that DOES perform I/O: sample_table()
# -----------------------------------------------------------------------------
# This call reads the TSV, validates required columns, filters DIV/sample IDs,
# derives matrix_dir paths, and returns a filtered metadata table.
sample_map = builder.sample_table()
print(f"Selected {len(sample_map)} samples from: {builder.sample_map_tsv}")
display(sample_map[["DIV", "run_sample_id", "biological_label", "matrix_dir"]])

Selected 6 samples from: /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline/metadata/div30_div90_sample_id_to_biolabel_map.tsv


,DIV,run_sample_id,biological_label,matrix_dir
0,DIV30,9853-MW-1,H9_rep1,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
1,DIV30,9853-MW-2,H9_rep2,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
2,DIV30,9853-MW-3,79B_rep1,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
3,DIV30,9853-MW-4,79B_rep2,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
4,DIV30,9853-MW-5,2E_rep1,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
5,DIV30,9853-MW-6,2E_rep2,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...


In [5]:
# Build the standard combined AnnData object for downstream Scanpy work.
sample_adata = builder.combined_anndata()

print("Combined AnnData shape:", sample_adata.shape)
print("Loaded run_sample_id count:", sample_adata.obs["run_sample_id"].nunique())
print("Unique cell IDs:", sample_adata.obs_names.is_unique)


Combined AnnData shape: (107020, 18082)
Loaded run_sample_id count: 6
Unique cell IDs: True


In [ ]:
# Multisample sanity checks without builder.cell_count_tables().
# (That helper was removed to keep the class minimal for this tutorial.)
sample_cell_counts = (
    sample_adata.obs.groupby(["DIV", "run_sample_id", "biological_label"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values(["DIV", "run_sample_id"])
    .reset_index(drop=True)
)

div_cell_counts = (
    sample_adata.obs.groupby("DIV", observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)

print("obs columns:", list(sample_adata.obs.columns))
print("Samples loaded:", sample_cell_counts["run_sample_id"].nunique())
display(sample_cell_counts)
display(div_cell_counts)

obs columns: ['DIV', 'run_sample_id', 'biological_label', 'matrix_dir']
Samples loaded: 6


,DIV,run_sample_id,biological_label,n_cells
0,DIV30,9853-MW-1,H9_rep1,18047
1,DIV30,9853-MW-2,H9_rep2,6060
2,DIV30,9853-MW-3,79B_rep1,17928
3,DIV30,9853-MW-4,79B_rep2,13047
4,DIV30,9853-MW-5,2E_rep1,26408
5,DIV30,9853-MW-6,2E_rep2,25530


,DIV,n_cells
0,DIV30,107020


In [7]:
# Quick object preview
sample_adata


AnnData object with n_obs × n_vars = 107020 × 18082
    obs: 'DIV', 'run_sample_id', 'biological_label', 'matrix_dir'
    var: 'gene_ids', 'feature_types'
    uns: 'sample_map_tsv', 'target_divs', 'combine_logic'

## First-Pass QC Metrics

This section computes standard Scanpy QC metrics on the combined DIV30 object and then summarizes them by `run_sample_id`. At this stage, `run_sample_id` is the batch/sample key, so per-sample QC is the right first view before filtering cells.


In [8]:
# Compute per-cell QC metrics once on the combined object.
# These columns are added to sample_adata.obs and can be summarized by sample afterward.
QC_GROUP_KEY = "run_sample_id"
QC_METRICS = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]

# Mark mitochondrial genes. Human gene symbols usually use MT- prefixes.
sample_adata.var["mt"] = sample_adata.var_names.str.upper().str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    sample_adata,
    qc_vars=["mt"],
    percent_top=[20, 50, 100],
    log1p=True,
    inplace=True,
)

print("Added QC columns to sample_adata.obs:")
display(sample_adata.obs[[QC_GROUP_KEY, "biological_label", *QC_METRICS]].head())


Added QC columns to sample_adata.obs:


,run_sample_id,biological_label,total_counts,n_genes_by_counts,pct_counts_mt
9853-MW-1:AAACAAGCAAGATAAGACTTTAGG-1,9853-MW-1,H9_rep1,8408.0,4355,0.047574
9853-MW-1:AAACAAGCAAGCCTAAACTTTAGG-1,9853-MW-1,H9_rep1,2046.0,1440,0.830890
9853-MW-1:AAACAAGCAAGGCCATACTTTAGG-1,9853-MW-1,H9_rep1,4643.0,2659,0.990739
9853-MW-1:AAACAAGCAATATGGTACTTTAGG-1,9853-MW-1,H9_rep1,6359.0,3742,1.320962
9853-MW-1:AAACAAGCACTAACGAACTTTAGG-1,9853-MW-1,H9_rep1,6076.0,3565,0.773535
